# V2 Pipeline - Phase 2: Huấn luyện Mô hình, Hyperparameter Tuning & Đánh giá Y tế (PTB-XL)

Thực hiện đầy đủ:
1. **Phân loại Nhị phân (Binary ML)**: Fine-tune **Logistic Regression** & **XGBoost** (khống chế `max_depth` 3-5 chống overfitting).
2. **Deep Learning (Neural Network / MLP)**: Tuning Hyperparameters `hidden_layer_sizes=(128, 64, 32)`.
3. **Foundation AI Model (TabPFN)**: Huấn luyện `TabPFNClassifier`.
4. **Tổ hợp Mô hình (Soft Voting & Stacking Ensemble)**.
5. **Đánh giá Y tế Chuyên sâu**: **Recall**, **F1-Score**, **ROC-AUC**, **Specificity**, **Accuracy**, và **False Negative Count (FN)**.

In [ ]:
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
import pyarrow
try:
    from tabpfn_client import TabPFNClassifier, init
    init()
    HAS_TABPFN = True
except Exception as e:
    HAS_TABPFN = False

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb, lightgbm as lgb, joblib
print('✅ Nạp thành công thư viện Training V2 cho PTB-XL!')


### 1. Nạp Dữ Liệu PTB-XL V2

In [ ]:
minmax_fname = 'v2_ptbxl_minmax_scaled.csv'
zscore_fname = 'v2_ptbxl_zscore_scaled.csv'
data_dir_candidates = ['../../../data/processed/ptbxl_processed', '../../../data/ptbxl_processed', '../../../data/features', '../../data/features', 'data/features']
data_dir = next((d for d in data_dir_candidates if os.path.exists(os.path.join(d, minmax_fname))), '../../data/features')
df_minmax = pd.read_csv(os.path.join(data_dir, minmax_fname))
df_zscore = pd.read_csv(os.path.join(data_dir, zscore_fname))
X_mm = df_minmax.drop(columns=['status'])
y_mm = df_minmax['status']
X_zs = df_zscore.drop(columns=['status'])
y_zs = df_zscore['status']
Xmm_train, Xmm_test, y_train, y_test = train_test_split(X_mm, y_mm, test_size=0.2, random_state=42, stratify=y_mm)
Xzs_train, Xzs_test, _, _ = train_test_split(X_zs, y_zs, test_size=0.2, random_state=42, stratify=y_zs)
print(f'Tập Train PTB-XL: {len(y_train)} mẫu | Tập Test: {len(y_test)} mẫu')


### 2. Fine-Tuning Mô Hình Binary ML & Deep Learning MLP (PTB-XL)

In [ ]:
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), {'C': [0.01, 0.1, 1.0, 10.0], 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear']}, cv=3, n_jobs=-1, scoring='f1')
grid_lr.fit(Xzs_train, y_train)
best_lr = grid_lr.best_estimator_

grid_xgb = GridSearchCV(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), {'max_depth': [3, 4, 5], 'learning_rate': [0.01, 0.05, 0.1], 'n_estimators': [100, 200, 300], 'subsample': [0.8, 1.0]}, cv=3, n_jobs=-1, scoring='f1')
grid_xgb.fit(Xmm_train, y_train)
best_xgb = grid_xgb.best_estimator_

grid_mlp = GridSearchCV(MLPClassifier(solver='adam', activation='logistic', random_state=42, max_iter=100, early_stopping=False), {'hidden_layer_sizes': [(32, 16, 8), (32, 24, 16, 8), (32, 24, 16, 8, 4)], 'learning_rate_init': [0.001, 0.01], 'batch_size': [32, 64]}, cv=3, n_jobs=-1, scoring='f1')
grid_mlp.fit(Xmm_train, y_train)
best_mlp = grid_mlp.best_estimator_
print('✅ Hoàn tất Fine-tuning Logistic Regression, XGBoost & Deep Learning MLP!')


### 3. Huấn luyện TabPFN & Ensemble Learning (PTB-XL)

In [ ]:
tabpfn_model = None
if HAS_TABPFN:
    try:
        clf_tab = TabPFNClassifier()
        clf_tab.fit(Xmm_train.values, y_train.values)
        tabpfn_model = clf_tab
        print('✅ Fit thành công TabPFN!')
    except Exception as ex:
        print(f'❌ TabPFN lỗi: {ex}')

svm_model = SVC(probability=True, random_state=42).fit(Xzs_train, y_train)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42).fit(Xmm_train, y_train)
et_model = ExtraTreesClassifier(n_estimators=200, max_depth=5, random_state=42).fit(Xmm_train, y_train)
lgb_model = lgb.LGBMClassifier(random_state=42, verbose=-1, n_estimators=200, learning_rate=0.05, max_depth=5).fit(Xmm_train, y_train)
knn_model = KNeighborsClassifier(n_neighbors=5).fit(Xmm_train, y_train)

voting_clf = VotingClassifier(
    estimators=[('svm', svm_model), ('xgb', best_xgb), ('rf', rf_model), ('mlp', best_mlp)],
    voting='soft', weights=[1.5, 2.0, 1.0, 1.0]
).fit(Xmm_train, y_train)
print('✅ Huấn luyện thành công Soft Voting Ensemble!')

stacking_clf = StackingClassifier(
    estimators=[('xgb', best_xgb), ('mlp', best_mlp), ('svm', svm_model), ('et', et_model), ('rf', rf_model)],
    final_estimator=LogisticRegression(random_state=42, max_iter=1000),
    cv=5,
    n_jobs=-1
).fit(Xmm_train, y_train)
print('✅ Huấn luyện thành công Stacking Ensemble (Meta-Learner: Logistic Regression)!')


### 4. Bảng Đánh Giá Tổng Hợp Thước Đo Y Tế (PTB-XL)

In [ ]:
all_models = {
    'Logistic Regression (Tuned)': (best_lr, Xzs_test),
    'XGBoost (Tuned)': (best_xgb, Xmm_test),
    'Neural Network / MLP (Nodes 8-32)': (best_mlp, Xmm_test),
    'Soft Voting Ensemble': (voting_clf, Xmm_test),
    'Stacking Ensemble (Meta-Learner)': (stacking_clf, Xmm_test)
}
if tabpfn_model is not None:
    all_models['TabPFN'] = (tabpfn_model, Xmm_test.values)

eval_results = []
roc_data = {}
conf_matrices = {}

for name, (model, X_test_set) in all_models.items():
    y_pred = model.predict(X_test_set)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_set)[:, 1]
    else:
        y_prob = y_pred
        
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_prob)
    
    eval_results.append({
        'Model': name,
        'Recall (Sensitivity)': rec,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Accuracy': acc,
        'Precision': prec,
        'Specificity': spec,
        'False Negative (FN)': fn
    })
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, roc_auc)
    conf_matrices[name] = cm

df_eval = pd.DataFrame(eval_results).sort_values(by=['Recall (Sensitivity)', 'F1-Score'], ascending=False).reset_index(drop=True)
badges = ['🥇 ', '🥈 ', '🥉 '] + [''] * (len(df_eval) - 3)
df_eval['Rank'] = [f'{badges[i]}#{i+1}' for i in range(len(df_eval))]
df_eval = df_eval[['Rank', 'Model', 'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC', 'Accuracy', 'Precision', 'Specificity', 'False Negative (FN)']]

styled_df = df_eval.style.format({
    'Recall (Sensitivity)': '{:.2%}', 'F1-Score': '{:.2%}', 'ROC-AUC': '{:.4f}',
    'Accuracy': '{:.2%}', 'Precision': '{:.2%}', 'Specificity': '{:.2%}', 'False Negative (FN)': '{:d}'
}).background_gradient(cmap='Blues', subset=['Recall (Sensitivity)', 'F1-Score', 'ROC-AUC'])

from IPython.display import display, HTML
display(HTML("<h2 style='text-align: center; color: #0f172a;'>🏆 BẢNG XẾP HẠNG THƯỚC ĐO Y TẾ - V2</h2>"))
display(styled_df)


### 5. Trực quan hóa Đường Cong ROC & Confusion Matrices (PTB-XL)

In [ ]:
plt.figure(figsize=(10, 8))
for name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.title('So sánh Đường Cong ROC Dữ Liệu PTB-XL V2')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()


### 6. Lưu trữ Models Trained V2 (PTB-XL)

In [ ]:
models_v2_dir_candidates = ['../../../models/v2/ptbxl', '../../models/v2/ptbxl', 'models/v2/ptbxl']
models_v2_dir = next((d for d in models_v2_dir_candidates if os.path.exists(os.path.dirname(d))), '../../models/v2/ptbxl')
os.makedirs(models_v2_dir, exist_ok=True)
joblib.dump(best_lr, os.path.join(models_v2_dir, 'logistic_regression_v2.pkl'))
joblib.dump(best_xgb, os.path.join(models_v2_dir, 'xgboost_v2.pkl'))
joblib.dump(best_mlp, os.path.join(models_v2_dir, 'neural_network_mlp_v2.pkl'))
joblib.dump(voting_clf, os.path.join(models_v2_dir, 'voting_ensemble_v2.pkl'))
joblib.dump(stacking_clf, os.path.join(models_v2_dir, 'stacking_ensemble_v2.pkl'))
print(f'✅ Đã lưu thành công các mô hình V2 vào: {models_v2_dir}')
